# 05 - Engine direct: build, check, run, plot

This notebook drives the ShakerMaker FK engine end to end on a tiny problem:

1. **Build** the model (SCEC LOH.1 crust + one Gaussian point source + a few
   surface stations).
2. **Check** the run parameters with `model.check_parameters(...)` -- pure
   arithmetic, no FK run, so it is the right place to validate `dt`, `nfft`,
   `dk`, `tb` and `tmax` *before* paying for a run.
3. **Run** a small synthetic with `model.run(...)`.
4. **Plot** the three-component response of one station with `ZENTPlot` and
   save it as a `.png`.

All units are in **km** (1 m = 1e-3 km).

In [ ]:
import matplotlib.pyplot as plt

from shakermaker.shakermaker import ShakerMaker
from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.tools.plotting import ZENTPlot

## 1. Build the model

`SCEC_LOH_1` is the canonical LOH.1 layered crust (a 1 km slow layer over a
half-space). The source is a Gaussian point source 2 km deep with a vertical
strike-slip mechanism `(strike, dip, rake) = (0, 90, 0)`. We record at three
free-surface stations near (6, 8) km.

In [ ]:
crust = SCEC_LOH_1()

sigma = 0.06
t0 = 6 * sigma
M0 = 1e18 / 5e14 / 2
stf = Gaussian(t0=t0, freq=1 / sigma, M0=M0, derivative=False)

source = PointSource([0, 0, 2], [0., 90., 0.], stf=stf)
fault = FaultSource([source], metadata={"name": "src"})

s1 = Station([6.0, 8.0, 0.0], metadata={"name": "s1"})
s2 = Station([8.0, 8.0, 0.0], metadata={"name": "s2"})
s3 = Station([6.0, 6.0, 0.0], metadata={"name": "s3"})
stations = StationList([s1, s2, s3], {})

model = ShakerMaker(crust, fault, stations)
print(crust)

## 2. Check the parameters

`check_parameters` is pure arithmetic: it mirrors what the FK core actually
uses and returns a dict with `passed` plus the *recommended* values for `dk`,
`tb`, `nfft` and `tmax`. Always call it right after building the model.

In [ ]:
dt, nfft, dk, tb, tmax = 0.025, 2048, 0.1, 1000, 30.
res = model.check_parameters(dt=dt, nfft=nfft, dk=dk, tb=tb, tmax=tmax)
res

## Preview the model before running
Before executing, we inspect the crust (layered column + velocity profile), the source geometry, and its source time function (STF). Each figure is saved as a PNG in this folder.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.tools.plotting import SourcePlot

crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt          # sample the STF before plotting the source
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## 3. Run a tiny synthetic

`model.run(...)` performs the FK integration and convolves with the source
time function, filling each station's response in place.

In [ ]:
model.run(dt=dt, nfft=nfft, tb=tb, dk=dk, tmax=tmax, verbose=False)

z, e, n, t = s1.get_response()
print("samples:", len(t), "  t range:", t[0], "->", t[-1])

## 4. Plot the station response

`ZENTPlot` draws the Z (down), E (east) and N (north) components. We save the
figure as a `.png` in this folder.

In [ ]:
ZENTPlot(s1, show=False)
plt.gcf().savefig("engine_direct_s1.png", dpi=150, bbox_inches="tight")